In [ ]:
from pathlib import Path

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "python").is_dir() and (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from the "
        "credit-risk-analytics project folder or one of its subfolders."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
LOCAL_DATA_DIR = PROJECT_ROOT / "local_data" / "processed"
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "local_outputs"

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_DIR)
print("Local processed-data folder:", LOCAL_DATA_DIR)
print("Local output folder:", LOCAL_OUTPUT_DIR)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

print("Train file found:", bool(train_matches))
print("Test file found:", bool(test_matches))
print("Column-description file found:", bool(columns_matches))
required_files = {"application_train.csv": train_matches, "application_test.csv": test_matches, "HomeCredit_columns_description.csv": columns_matches}
missing = [name for name, matches in required_files.items() if not matches]
if missing:
    raise FileNotFoundError(f"Missing dataset file(s) in {DATA_DIR}: {missing}. See the project README for data setup instructions.")


In [ ]:
train_path = train_matches[0]
test_path = test_matches[0]
columns_path = columns_matches[0]

print("Training file:", train_path)
print("Test file:", test_path)
print("Columns file:", columns_path)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
columns_desc = pd.read_csv(columns_path, encoding="latin1")

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)
print("Column description shape:", columns_desc.shape)

In [ ]:
train.head()

In [ ]:
print("Training data rows:", train.shape[0])
print("Training data columns:", train.shape[1])
print("Test data rows:", test.shape[0])
print("Test data columns:", test.shape[1])

In [ ]:
train["TARGET"].value_counts()

In [ ]:
default_rate = train["TARGET"].mean()
print(f"Default rate: {default_rate:.2%}")

In [ ]:
train["TARGET"].value_counts(normalize=True) * 100
print("The target variable is imbalanced, with approximately 91.93% non-default cases and 8.07% default cases. This is expected in credit risk data, as defaults are usually less frequent than non-defaults. The imbalance will be considered during modelling by using appropriate evaluation metrics such as ROC-AUC, precision, recall, F1-score and confusion matrices, rather than relying only on accuracy.")

In [ ]:
# Check data types and non-null counts
train.info()

In [ ]:
# Separate numerical and categorical columns

numeric_cols = train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = train.select_dtypes(include=["object"]).columns.tolist()

print("Number of numeric columns:", len(numeric_cols))
print("Number of categorical columns:", len(categorical_cols))

In [ ]:
# Missing values summary

missing_summary = pd.DataFrame({
    "Missing Count": train.isnull().sum(),
    "Missing Percentage": train.isnull().mean() * 100
})

missing_summary = missing_summary[missing_summary["Missing Count"] > 0]
missing_summary = missing_summary.sort_values(by="Missing Percentage", ascending=False)

missing_summary.head(20)

In [ ]:
# Overall missing data summary

total_missing_values = train.isnull().sum().sum()
total_cells = train.shape[0] * train.shape[1]
overall_missing_percentage = (total_missing_values / total_cells) * 100

print("Total missing values:", total_missing_values)
print(f"Overall missing percentage: {overall_missing_percentage:.2f}%")
print("Number of columns with missing values:", missing_summary.shape[0])
print("Total number of columns:", train.shape[1])

In [ ]:
# Group columns by missing percentage severity

high_missing = missing_summary[missing_summary["Missing Percentage"] >= 50]
medium_missing = missing_summary[
    (missing_summary["Missing Percentage"] >= 20) & 
    (missing_summary["Missing Percentage"] < 50)
]
low_missing = missing_summary[missing_summary["Missing Percentage"] < 20]

print("Columns with high missingness (>=50%):", high_missing.shape[0])
print("Columns with medium missingness (20%-50%):", medium_missing.shape[0])
print("Columns with low missingness (<20%):", low_missing.shape[0])

In [ ]:
print(""" The dataset contains substantial missing data. Overall, approximately 24.40% of all values are missing, with 67 out of 122 variables containing at least one missing value. The highest levels of missingness are observed in variables related to building or housing characteristics, such as COMMONAREA, NONLIVINGAPARTMENTS, FLOORSMIN, YEARS_BUILD and LANDAREA. Many of these variables have more than 50% missing values.

This suggests that missing data is a major feature of the credit risk dataset and should be handled carefully. Since missingness in financial data may contain useful information, variables will not be removed automatically. Instead, variables will be grouped based on the severity of missingness, and different strategies such as removal, imputation and missing-value indicators will be considered during modelling.""")

In [ ]:
# Compare missingness between non-defaulters and defaulters

missing_by_target = pd.DataFrame({
    "Missing % Non-Default": train[train["TARGET"] == 0].isnull().mean() * 100,
    "Missing % Default": train[train["TARGET"] == 1].isnull().mean() * 100
})

missing_by_target["Difference"] = (
    missing_by_target["Missing % Default"] - 
    missing_by_target["Missing % Non-Default"]
)

missing_by_target = missing_by_target.loc[missing_summary.index]
missing_by_target = missing_by_target.sort_values(by="Difference", ascending=False)

missing_by_target.head(20)

In [ ]:
# Variables where missingness differs most between defaulters and non-defaulters

missing_by_target_abs = missing_by_target.copy()
missing_by_target_abs["Absolute Difference"] = missing_by_target_abs["Difference"].abs()

missing_by_target_abs = missing_by_target_abs.sort_values(
    by="Absolute Difference", 
    ascending=False
)

missing_by_target_abs.head(20)

In [ ]:
print(""" Missingness differs between defaulters and non-defaulters for several variables. The largest differences are observed in housing and building-related variables such as EMERGENCYSTATE_MODE, TOTALAREA_MODE, ENTRANCES, FLOORSMAX, YEARS_BEGINEXPLUATATION, APARTMENTS, ELEVATORS and LIVINGAREA. For these variables, missingness is approximately 7 percentage points higher among applicants who defaulted.

This suggests that missing values may not be completely random. In a credit risk context, the absence of information may itself contain risk information. Therefore, missing values should not be treated only as data quality problems. During modelling, missing-value indicators may be considered so that the model can capture whether missingness is associated with default risk.""")

In [ ]:
# Descriptive statistics for numerical variables

numeric_summary = train[numeric_cols].describe().T

numeric_summary["missing_percentage"] = train[numeric_cols].isnull().mean() * 100

numeric_summary.head(20)

In [ ]:
# Check variables with the largest maximum values

numeric_summary_sorted_max = numeric_summary.sort_values(by="max", ascending=False)

numeric_summary_sorted_max[["mean", "std", "min", "max", "missing_percentage"]].head(20)

In [ ]:
important_numeric_vars = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

train[important_numeric_vars].describe().T

In [ ]:
# Identify all variables that contain "DAYS"

days_cols = [col for col in train.columns if "DAYS" in col]

days_summary = train[days_cols].describe().T
days_summary["missing_percentage"] = train[days_cols].isnull().mean() * 100

days_summary

In [ ]:
# Create age in years

train["AGE_YEARS"] = (-train["DAYS_BIRTH"]) / 365.25

train["AGE_YEARS"].describe()

In [ ]:
# Check employment duration values

train["DAYS_EMPLOYED"].describe()

In [ ]:
# Show the largest DAYS_EMPLOYED values

train["DAYS_EMPLOYED"].value_counts().sort_index(ascending=False).head(10)

In [ ]:
# Summary of categorical variables

categorical_summary = pd.DataFrame({
    "Unique Values": train[categorical_cols].nunique(),
    "Missing Count": train[categorical_cols].isnull().sum(),
    "Missing Percentage": train[categorical_cols].isnull().mean() * 100
})

categorical_summary.sort_values(by="Unique Values", ascending=False)

In [ ]:
# Display unique values for each categorical variable

for col in categorical_cols:
    print(f"\n{col}")
    print(train[col].value_counts(dropna=False).head(20))

In [ ]:
# Create a cleaned copy of the training dataset

train_clean = train.copy()

# Create an anomaly flag for the strange DAYS_EMPLOYED value
train_clean["DAYS_EMPLOYED_ANOMALY"] = (train_clean["DAYS_EMPLOYED"] == 365243).astype(int)

# Replace the strange placeholder value with missing
train_clean["DAYS_EMPLOYED_CLEAN"] = train_clean["DAYS_EMPLOYED"].replace(365243, np.nan)

# Convert employment duration into years
train_clean["YEARS_EMPLOYED"] = (-train_clean["DAYS_EMPLOYED_CLEAN"]) / 365.25

# Check the cleaned employment variable
train_clean[["DAYS_EMPLOYED", "DAYS_EMPLOYED_ANOMALY", "DAYS_EMPLOYED_CLEAN", "YEARS_EMPLOYED"]].describe().T

In [ ]:
# Check how common the DAYS_EMPLOYED anomaly is by target group

employment_anomaly_by_target = train_clean.groupby("TARGET")["DAYS_EMPLOYED_ANOMALY"].mean() * 100

employment_anomaly_by_target

In [ ]:
print(""" The descriptive analysis identified a major anomaly in the DAYS_EMPLOYED variable. The value 365243 appears for 55,374 applicants and is not a realistic employment duration, as it would represent approximately 1,000 years of employment. This value is therefore treated as a placeholder rather than a valid numerical value.

To handle this, a binary indicator variable, DAYS_EMPLOYED_ANOMALY, was created to preserve the information that the value was originally abnormal. The value 365243 was then replaced with missing in a cleaned employment variable, and employment duration was converted into years for easier interpretation.

Other potential data quality issues were also identified, including an extreme maximum value in AMT_INCOME_TOTAL and rare categorical levels such as CODE_GENDER = XNA and NAME_FAMILY_STATUS = Unknown. These will be considered during later data cleaning and modelling steps.""")

In [ ]:
# Make sure train_clean exists
try:
    train_clean
except NameError:
    train_clean = train.copy()

# Age in years
if "AGE_YEARS" not in train_clean.columns:
    train_clean["AGE_YEARS"] = (-train_clean["DAYS_BIRTH"]) / 365.25

# Clean employment duration if not already created
if "DAYS_EMPLOYED_ANOMALY" not in train_clean.columns:
    train_clean["DAYS_EMPLOYED_ANOMALY"] = (train_clean["DAYS_EMPLOYED"] == 365243).astype(int)

if "DAYS_EMPLOYED_CLEAN" not in train_clean.columns:
    train_clean["DAYS_EMPLOYED_CLEAN"] = train_clean["DAYS_EMPLOYED"].replace(365243, np.nan)

if "YEARS_EMPLOYED" not in train_clean.columns:
    train_clean["YEARS_EMPLOYED"] = (-train_clean["DAYS_EMPLOYED_CLEAN"]) / 365.25

# Create useful credit risk ratios
train_clean["CREDIT_TO_INCOME_RATIO"] = train_clean["AMT_CREDIT"] / train_clean["AMT_INCOME_TOTAL"]
train_clean["ANNUITY_TO_INCOME_RATIO"] = train_clean["AMT_ANNUITY"] / train_clean["AMT_INCOME_TOTAL"]
train_clean["GOODS_TO_CREDIT_RATIO"] = train_clean["AMT_GOODS_PRICE"] / train_clean["AMT_CREDIT"]

In [ ]:
# Compare key numerical variables between non-defaulters and defaulters

key_numeric_vars = [
    "AGE_YEARS",
    "YEARS_EMPLOYED",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CREDIT_TO_INCOME_RATIO",
    "ANNUITY_TO_INCOME_RATIO",
    "GOODS_TO_CREDIT_RATIO",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "DAYS_EMPLOYED_ANOMALY"
]

numeric_target_summary = []

for col in key_numeric_vars:
    non_default = train_clean.loc[train_clean["TARGET"] == 0, col]
    default = train_clean.loc[train_clean["TARGET"] == 1, col]
    
    numeric_target_summary.append({
        "Variable": col,
        "Non_Default_Mean": non_default.mean(),
        "Default_Mean": default.mean(),
        "Mean_Difference": default.mean() - non_default.mean(),
        "Non_Default_Median": non_default.median(),
        "Default_Median": default.median(),
        "Missing_%": train_clean[col].isnull().mean() * 100
    })

numeric_target_summary = pd.DataFrame(numeric_target_summary)

numeric_target_summary.sort_values(by="Mean_Difference", ascending=False)

In [ ]:
numeric_target_summary_display = numeric_target_summary.copy()

for col in [
    "Non_Default_Mean", 
    "Default_Mean", 
    "Mean_Difference", 
    "Non_Default_Median", 
    "Default_Median", 
    "Missing_%"
]:
    numeric_target_summary_display[col] = numeric_target_summary_display[col].round(4)

numeric_target_summary_display

In [ ]:
# Function to calculate default rate by category

def default_rate_by_category(data, column, min_count=100):
    summary = data.groupby(column, dropna=False).agg(
        Applicant_Count=("TARGET", "count"),
        Default_Count=("TARGET", "sum"),
        Default_Rate=("TARGET", "mean")
    ).reset_index()
    
    summary["Default_Rate_%"] = summary["Default_Rate"] * 100
    summary = summary[summary["Applicant_Count"] >= min_count]
    summary = summary.sort_values(by="Default_Rate_%", ascending=False)
    
    return summary[[column, "Applicant_Count", "Default_Count", "Default_Rate_%"]]

In [ ]:
important_categorical_vars = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE"
]

for col in important_categorical_vars:
    print("\n" + "="*80)
    print(f"Default rate by {col}")
    print("="*80)
    display(default_rate_by_category(train_clean, col).head(15))

In [ ]:
important_categorical_vars = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE"
]

for col in important_categorical_vars:
    print("\n" + "="*80)
    print(f"Default rate by {col}")
    print("="*80)
    display(default_rate_by_category(train_clean, col).head(15))

In [ ]:
print(""" ## Bivariate Analysis: Comparing Defaulters and Non-Defaulters

The next step is to compare applicant characteristics between defaulters and non-defaulters. This helps identify variables that may be associated with default risk before formal modelling is performed. Numerical variables are compared using means and medians, while categorical variables are compared using default rates by category.

This analysis is exploratory and does not prove causality. However, it provides useful insight into potential risk drivers and helps guide feature selection for later statistical and machine learning models.""")

In [ ]:
print(""" The bivariate analysis shows clear differences between defaulters and non-defaulters. Defaulters are younger on average, have shorter employment histories, slightly lower incomes, and lower external credit scores. The external score variables show particularly strong separation between default and non-default applicants, suggesting that they may be important predictors in later modelling.

Categorical variables also show meaningful differences in default rates. Higher default rates are observed among applicants with lower education levels, rented housing, certain occupation types such as low-skill laborers and drivers, and certain organisation types such as transport, restaurant and construction. These results are exploratory and do not imply causality, but they provide useful insight into potential credit risk drivers.""")

In [ ]:
# Function to calculate default rate by numerical bins

def default_rate_by_numeric_bins(data, column, bins=5):
    temp = data[[column, "TARGET"]].dropna().copy()
    
    temp[f"{column}_BIN"] = pd.qcut(
        temp[column], 
        q=bins, 
        duplicates="drop"
    )
    
    summary = temp.groupby(f"{column}_BIN", observed=True).agg(
        Applicant_Count=("TARGET", "count"),
        Default_Count=("TARGET", "sum"),
        Default_Rate=("TARGET", "mean")
    ).reset_index()
    
    summary["Default_Rate_%"] = summary["Default_Rate"] * 100
    
    return summary

In [ ]:
numeric_bin_vars = [
    "AGE_YEARS",
    "YEARS_EMPLOYED",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "ANNUITY_TO_INCOME_RATIO"
]

for col in numeric_bin_vars:
    print("\n" + "="*80)
    print(f"Default rate by {col} bands")
    print("="*80)
    display(default_rate_by_numeric_bins(train_clean, col))

In [ ]:
print(""" The binned analysis provides stronger evidence of risk patterns in the data. Default risk decreases consistently with age, employment duration and external source scores. The external source variables appear to be especially powerful, with default rates falling from above 14% in the lowest score bands to around 3% in the highest score bands. This indicates that these variables are likely to be important in later predictive modelling.

The annuity-to-income ratio shows a weaker but still relevant affordability pattern, with slightly higher default rates among applicants with higher repayment burden. Overall, the bivariate analysis suggests that applicant stability, external credit scores and affordability measures should be prioritised during feature selection.""")

In [ ]:
## Key Findings from Data Understanding
print("""The dataset contains 307,511 training observations and 122 variables. The target variable is imbalanced, with approximately 91.93% non-default cases and 8.07% default cases. This is expected in credit risk data, where defaults are usually less frequent than non-defaults. Because of this imbalance, accuracy alone will not be sufficient for model evaluation. Metrics such as ROC-AUC, precision, recall, F1-score and confusion matrices should be considered during modelling.

Missing data is a major feature of the dataset. Approximately 24.40% of all values are missing, and 67 out of 122 variables contain at least one missing value. Several housing and building-related variables have more than 50% missingness. Missingness also differs between defaulters and non-defaulters for some variables, suggesting that missing values may contain useful risk information rather than being purely random data quality issues.

The descriptive analysis identified a major anomaly in the DAYS_EMPLOYED variable, where the value 365243 appears for 55,374 applicants. This value is not realistic and should be treated as a placeholder. A cleaned employment duration variable and an anomaly indicator should be created during feature engineering.

The bivariate analysis showed meaningful differences between defaulters and non-defaulters. Defaulters are generally younger, have shorter employment histories, slightly lower incomes, and lower external credit scores. The EXT_SOURCE variables appear especially important, with default rates decreasing strongly as external scores increase.

Categorical analysis also showed important risk patterns. Higher default rates were observed among applicants with lower education levels, rented housing, certain occupation types such as low-skill laborers and drivers, and certain organisation types such as transport, restaurant and construction. These findings are exploratory and do not imply causality, but they provide useful direction for feature selection and modelling.

Overall, the data understanding stage shows that applicant stability, external credit scores, affordability measures, missing data patterns and employment-related variables are likely to be important for predicting loan default risk.""")